## top 1. 读取csv数据文件

In [0]:
csv_path = "/Volumes/mycat/market/myvol/sales.csv"
# 读取 CSV 
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

df.show()

## top 2. 保存读取的数据到 Delta 表

In [0]:

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mycat.market.sales")

## top 3. 查看表结构describe

In [0]:
%sql
describe extended mycat.market.sales;

col_name,data_type,comment
ROW_ID,int,null
ORDER_ID,string,null
ORDER_DATE,date,null
SHIP_DATE,date,null
SHIP_METHOD,string,null
CUSTOMER_ID,string,null
CUSTOMER_NAME,string,null
CUST_CATEGORY,string,null
CITY,string,null
PROVINCE,string,null


## top 4. 从delta table读取数据

In [0]:
df = spark.table("mycat.market.sales");
df.show()

+------+-----------------+----------+----------+-----------+-----------+-------------+-------------+------+--------+-------+------+--------------------+--------+------------+--------------------------+---------+--------+--------+---------+
|ROW_ID|         ORDER_ID|ORDER_DATE| SHIP_DATE|SHIP_METHOD|CUSTOMER_ID|CUSTOMER_NAME|CUST_CATEGORY|  CITY|PROVINCE|COUNTRY|REGION|          PRODUCT_ID|CATEGORY|SUB_CATEGORY|              PRODUCT_NAME|    SALES|QUANTITY|DISCOUNT|   PROFIT|
+------+-----------------+----------+----------+-----------+-----------+-------------+-------------+------+--------+-------+------+--------------------+--------+------------+--------------------------+---------+--------+--------+---------+
|     1|US-202023-1357144|2024-04-27|2024-04-29|       二级| 曾惠-14485|         曾惠|         公司|  杭州|    浙江|   中国|  华东|办公用-用品-10002717|办公用品|        用品|        Fiskars 剪刀, 蓝色|  129.696|       2|     0.4|  -60.704|
|     2|CN-202023-1973789|2024-06-15|2024-06-19|     标准级| 许安-10165|    

## top 5. select操作：选择特定列

In [0]:
df_cities = df.select("CITY", "PROVINCE", "COUNTRY")
df_cities.show()

+------+--------+-------+
|  CITY|PROVINCE|COUNTRY|
+------+--------+-------+
|  杭州|    浙江|   中国|
|  内江|    四川|   中国|
|  内江|    四川|   中国|
|  镇江|    江苏|   中国|
|  汕头|    广东|   中国|
|景德镇|    江西|   中国|
|景德镇|    江西|   中国|
|景德镇|    江西|   中国|
|景德镇|    江西|   中国|
|景德镇|    江西|   中国|
|  榆林|    陕西|   中国|
|哈尔滨|  黑龙江|   中国|
|  青岛|    山东|   中国|
|  青岛|    山东|   中国|
|  青岛|    山东|   中国|
|  徐州|    江苏|   中国|
|  徐州|    江苏|   中国|
|  上海|    上海|   中国|
|  上海|    上海|   中国|
|  上海|    上海|   中国|
+------+--------+-------+
only showing top 20 rows


## top 6. filter操作：过滤行

In [0]:

df_sc = df.filter(df.PROVINCE == "四川").select("PROVINCE","ORDER_ID")
df_sc.show()

+--------+-----------------+
|PROVINCE|         ORDER_ID|
+--------+-----------------+
|    四川|CN-202023-1973789|
|    四川|CN-202023-1973789|
|    四川|US-202022-5956361|
|    四川|US-202022-5956361|
|    四川|US-202022-5956361|
|    四川|CN-202022-5552260|
|    四川|CN-202023-5816959|
|    四川|CN-202023-5816959|
|    四川|CN-202023-5816959|
|    四川|US-202020-3069391|
|    四川|US-202020-3069391|
|    四川|CN-202020-3174894|
|    四川|US-202021-1388500|
|    四川|US-202020-4650370|
|    四川|CN-202022-1424056|
|    四川|CN-202022-1424056|
|    四川|US-202023-1184235|
|    四川|CN-202023-3904829|
|    四川|CN-202022-3767771|
|    四川|CN-202023-4195213|
+--------+-----------------+
only showing top 20 rows


## top 7. withcolumn操作：增加新列

In [0]:
from pyspark.sql.functions import lit

df_newcol = df_sc.withColumn("NEW_PROVINCE", lit("sichuan"))
df_newcol.show()


+--------+-----------------+------------+
|PROVINCE|         ORDER_ID|NEW_PROVINCE|
+--------+-----------------+------------+
|    四川|CN-202023-1973789|     sichuan|
|    四川|CN-202023-1973789|     sichuan|
|    四川|US-202022-5956361|     sichuan|
|    四川|US-202022-5956361|     sichuan|
|    四川|US-202022-5956361|     sichuan|
|    四川|CN-202022-5552260|     sichuan|
|    四川|CN-202023-5816959|     sichuan|
|    四川|CN-202023-5816959|     sichuan|
|    四川|CN-202023-5816959|     sichuan|
|    四川|US-202020-3069391|     sichuan|
|    四川|US-202020-3069391|     sichuan|
|    四川|CN-202020-3174894|     sichuan|
|    四川|US-202021-1388500|     sichuan|
|    四川|US-202020-4650370|     sichuan|
|    四川|CN-202022-1424056|     sichuan|
|    四川|CN-202022-1424056|     sichuan|
|    四川|US-202023-1184235|     sichuan|
|    四川|CN-202023-3904829|     sichuan|
|    四川|CN-202022-3767771|     sichuan|
|    四川|CN-202023-4195213|     sichuan|
+--------+-----------------+------------+
only showing top 20 rows


## top 8. groupby和agg操作：分组聚合

In [0]:

from pyspark.sql.functions import count, sum, avg, min, max

df_profit = df.groupBy("PROVINCE").agg(count("ORDER_ID").alias("Frequency"), sum("PROFIT").alias("Monetary"), avg("PROFIT").alias("Average"), min("PROFIT").alias("Min"), max("PROFIT").alias("Max"))
df_profit.show()


+--------+---------+-------------------+-------------------+---------+--------+
|PROVINCE|Frequency|           Monetary|            Average|      Min|     Max|
+--------+---------+-------------------+-------------------+---------+--------+
|    浙江|      415|-131728.99600000013|-317.41926746987986|-6908.496|  948.36|
|    四川|      392| -89487.52399999995|-228.28449999999987|-6748.224| 1097.04|
|    江苏|      578|-107603.02000000003| -186.1643944636679| -7978.32| 2845.08|
|    广东|      834|  337994.9929999998| 405.26977577937623| -2520.35| 8407.56|
|    江西|      139|  47807.06000000001|  343.9356834532375|-4515.252| 3783.78|
|    陕西|      236| 105814.68799999998| 448.36732203389823| -169.036| 5653.76|
|  黑龙江|      636| 257172.06199999977|  404.3585880503141|-1479.555| 7214.76|
|    山东|      914|         385463.008|  421.7319562363238| -5294.52|  5623.8|
|    上海|      292|         121650.088| 416.60989041095894|-1881.824|  6115.2|
|    河北|      388|  172031.6850000001| 443.38063144329925|-

## top 9. join操作：连接两个DataFrame

In [0]:
# 先把23,24年销售数据导入sales_2023，sales_2024两张表
csv_path = "/Volumes/mycat/market/myvol/sales_2023.csv"
# 读取 CSV 
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)


df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mycat.market.sales_2023")

csv_path = "/Volumes/mycat/market/myvol/sales_2024.csv"
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)


df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mycat.market.sales_2024")

In [0]:
df_2024 = spark.table("mycat.market.sales_2024");
df_2023 = spark.table("mycat.market.sales_2023");

customer_in_2023 = df_2023.select("customer_id").distinct()
customer_in_2024 = df_2024.select("customer_id").distinct()

customer_23_24 = customer_in_2023.join(customer_in_2024, "customer_id", "inner").count()
customer_23_24


583

## top 10. orderBy操作：排序

In [0]:
from pyspark.sql import functions as F

df_orderby = spark.read.table("mycat.market.sales")
df_result = (
    df.groupBy("REGION")
      .agg(
          F.avg("SALES").alias("average_sales"),
          F.avg("PROFIT").alias("average_profit")
      )
      .orderBy(F.col("average_profit").desc())
)
df_result.show()

+------+------------------+------------------+
|REGION|     average_sales|    average_profit|
+------+------------------+------------------+
|  华北| 1838.483691176469| 297.1601617647061|
|  中南|1563.2958914728697|252.69589147286814|
|  华东|1601.3148089887611|185.62794484167517|
|  西北|1805.8374133333343|176.21034666666665|
|  西南|1508.1936406250006|119.02887499999991|
|  东北| 1551.392460032628|118.84029037520398|
+------+------------------+------------------+



## 11. write操作：将DataFrame写入delta table

In [0]:

df_result.write.mode("overwrite").saveAsTable("mycat.market.result")

df_result.show()


+------+------------------+------------------+
|REGION|     average_sales|    average_profit|
+------+------------------+------------------+
|  华北| 1838.483691176469| 297.1601617647061|
|  中南|1563.2958914728697|252.69589147286814|
|  华东|1601.3148089887611|185.62794484167517|
|  西北|1805.8374133333343|176.21034666666665|
|  西南|1508.1936406250006|119.02887499999991|
|  东北| 1551.392460032628|118.84029037520398|
+------+------------------+------------------+

